# Inspect cam03 POC_2_TAGS Artifacts
This notebook verifies the Python environment, then loads and summarizes Stage D artifacts for the clip `cam03-20260103-124000_0-30s`, focusing on POC_2_TAGS tag bindings and labeled SOLO nodes.

## 1. Set Up Python Environment in Notebook
This section imports core libraries and prints basic version information to verify the Python environment.

In [1]:
# 1. Set Up Python Environment in Notebook
import sys
import os
import platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

Python executable: /Users/bryanthomas/miniconda3/envs/bjj_env/bin/python
Python version: 3.10.17 | packaged by conda-forge | (main, Apr 10 2025, 22:23:34) [Clang 18.1.8 ]
Platform: macOS-26.2-arm64-arm-64bit


## 2. Verify Notebook Execution
Run a few simple Python statements to confirm that notebook cells execute correctly.

In [2]:
# 2. Verify Notebook Execution
print("2 + 2 =", 2 + 2)
vals = [1, 2, 3]
print("Sum of vals =", sum(vals))

2 + 2 = 4
Sum of vals = 6


## 3. Define and Call a Simple Function
Define a Python function and call it with a few arguments to verify stateful execution.

In [3]:
# 3. Define and Call a Simple Function
def square(x: float) -> float:
    return x * x

for v in [0, 1, 2, 3, 4]:
    print(f"square({v}) =", square(v))

square(0) = 0
square(1) = 1
square(2) = 4
square(3) = 9
square(4) = 16


## 4. Work with the File System from the Notebook
Use the `os` module to inspect the current working directory and list files in the notebooks folder.

In [5]:
# 4. Work with the File System from the Notebook
import os
from pathlib import Path

cwd = Path.cwd()
print("Current working directory:", cwd)

notebooks_dir = cwd
print("\nFiles in current directory:")
for p in sorted(notebooks_dir.iterdir()):
    print(" -", p.name)

Current working directory: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/notebooks

Files in current directory:
 - d2_ms_vs_cont_term_vreq.ipynb
 - d3_stitch_eval_cam03.ipynb
 - inspect_cam03_poc2_tags.ipynb
 - temp.ipynb


## 5. Run a Shell Command from the Notebook
Use a simple shell command to verify basic integration with the OS.

In [6]:
# 5. Run a Shell Command from the Notebook
!ls -1

d2_ms_vs_cont_term_vreq.ipynb
d3_stitch_eval_cam03.ipynb
inspect_cam03_poc2_tags.ipynb
temp.ipynb


## 6. Analyze cam03 POC_2_TAGS Artifacts
Load D1/D2/D3 artifacts for `cam03-20260103-124000_0-30s` and summarize tag bindings and labeled SOLO nodes.

In [9]:
# 6. Analyze cam03 POC_2_TAGS Artifacts
from pathlib import Path
import json
import pandas as pd

# Notebook runs with CWD = project_root/notebooks, so clip root is one level up.
CLIP_ROOT = Path("..") / "outputs" / "cam03-20260103-124000_0-30s"
STAGE_D = CLIP_ROOT / "stage_D"
DEBUG_DIR = CLIP_ROOT / "_debug"

TIDS = ["t5", "t16", "t2", "t15", "t3", "t14"]

def load_artifacts():
    nodes = pd.read_parquet(STAGE_D / "d1_graph_nodes.parquet")
    edges = pd.read_parquet(STAGE_D / "d1_graph_edges.parquet")
    costs = pd.read_parquet(STAGE_D / "d2_edge_costs.parquet")
    with (DEBUG_DIR / "d3_solution_ledger.json").open("r", encoding="utf-8") as f:
        ledger = json.load(f)
    tags = ledger.get("tags", {})
    return nodes, edges, costs, ledger, tags

def summarize_tags(tags: dict):
    print("\n=== TAGS SUMMARY ===")
    summary_keys = ["n_identity_hints", "n_tag_hints", "labels_domain"]
    summary = {k: tags.get(k) for k in summary_keys}
    print(json.dumps(summary, indent=2))

    binding_ledger = tags.get("binding_ledger", [])
    conflicts = [r for r in binding_ledger if r.get("status") == "conflict"]
    print(f"\nBinding ledger size: {len(binding_ledger)}")
    print(f"Conflicts in binding ledger: {len(conflicts)}")
    if conflicts:
        print("\nFirst few conflicts:")
        for r in conflicts[:5]:
            print(json.dumps(r, indent=2))

def summarize_solo_labels(nodes: pd.DataFrame, tags: dict):
    print("\n=== SOLO NODE LABELS USED ===")
    solo_labels = tags.get("solo_node_labels_used", [])
    print(f"Total labeled SOLO nodes (used): {len(solo_labels)}")
    if not solo_labels:
        return
    df_labels = pd.DataFrame(solo_labels)
    df_labels["node_id"] = df_labels["node_id"].astype(str)
    nodes = nodes.copy()
    nodes["node_id"] = nodes["node_id"].astype(str)
    if "base_tracklet_id" in nodes.columns:
        nodes["base_tracklet_id"] = nodes["base_tracklet_id"].astype(str)
    df = nodes.merge(df_labels, on="node_id", how="right")
    if "node_type" in df.columns:
        df = df[df["node_type"] == "NodeType.SINGLE_TRACKLET"]
    print("\nSample of labeled SOLO nodes (head 20):")
    cols = [c for c in ["node_id", "base_tracklet_id", "start_frame", "end_frame", "label"] if c in df.columns]
    print(df[cols].sort_values(["base_tracklet_id", "start_frame"]).head(20))
    if "base_tracklet_id" in df.columns:
        print("\nLabeled SOLO nodes for the 6 injected tids:")
        df_6 = df[df["base_tracklet_id"].isin(TIDS)].copy()
        print(df_6[cols].sort_values(["base_tracklet_id", "start_frame"]))

nodes, edges, costs, ledger, tags = load_artifacts()
print("Loaded:")
print(f"  d1_graph_nodes: {len(nodes)} rows")
print(f"  d1_graph_edges: {len(edges)} rows")
print(f"  d2_edge_costs:  {len(costs)} rows")

summarize_tags(tags)
summarize_solo_labels(nodes, tags)

Loaded:
  d1_graph_nodes: 33 rows
  d1_graph_edges: 72 rows
  d2_edge_costs:  72 rows

=== TAGS SUMMARY ===
{
  "n_identity_hints": 6,
  "n_tag_hints": 6,
  "labels_domain": [
    "tag:1",
    "tag:2",
    "tag:3",
    "tag:4",
    "UNKNOWN"
  ]
}

Binding ledger size: 6
Conflicts in binding ledger: 0

=== SOLO NODE LABELS USED ===
Total labeled SOLO nodes (used): 17

Sample of labeled SOLO nodes (head 20):
            node_id base_tracklet_id  start_frame  end_frame    label
5   T:t1:s1:138-280               t1          138        280    tag:1
0             T:t10              t10          508        547    tag:2
1             T:t11              t11          623        742    tag:4
2             T:t13              t13          758        810    tag:2
3             T:t14              t14          796        900    tag:4
4             T:t16              t16          853        900    tag:2
6              T:t2               t2           10        792    tag:3
7   T:t4:s0:138-280          